# GSU Derivation Cohort — END Rate by NIHSS Subset

Evaluate the END (Early Neurological Deterioration) rate in the GSU derivation cohort overall and when filtered to a NIHSS-severity subset comparable to the MIMIC external validation cohort (median NIHSS 13.0, IQR 7–19, END rate 56.3%).

In [ ]:
import os
import numpy as np
import pandas as pd
import torch

In [ ]:
DATA_DIR = '/mnt/data1/klug/datasets/opsum/short_term_outcomes/with_imaging/gsu_Extraction_20220815_prepro_30012026_154047'
TRAIN_PATH = os.path.join(DATA_DIR, 'train_data_splits_early_neurological_deterioration_ts0.8_rs42_ns5.pth')
TEST_PATH = os.path.join(DATA_DIR, 'test_data_early_neurological_deterioration_ts0.8_rs42_ns5.pth')
NORM_PARAMS_PATH = os.path.join(DATA_DIR, 'logs_30012026_154047/normalisation_parameters.csv')

N_BOOTSTRAP = 10000
RANDOM_SEED = 42

In [ ]:
# Load train splits (fold 0) and test set
print('Loading training data (fold 0)...')
train_splits = torch.load(TRAIN_PATH, map_location='cpu')
X_train, X_val, y_train, y_val = train_splits[0]
del train_splits

print('Loading test data...')
X_test, y_test = torch.load(TEST_PATH, map_location='cpu')

# Concatenate to get full GSU cohort
X_all = np.concatenate([X_train, X_val, X_test], axis=0)
y_all = pd.concat([y_train, y_val, y_test], ignore_index=True)
del X_train, X_val, X_test, y_train, y_val, y_test

# Load normalisation parameters
norm_params = pd.read_csv(NORM_PARAMS_PATH)

print(f'X_all shape: {X_all.shape}')
print(f'Total patients: {X_all.shape[0]}')
print(f'Unique END patients in y_all: {y_all["case_admission_id"].nunique()}')

In [ ]:
# Extract admission NIHSS
# Tensor shape: (n_patients, timesteps, features, 4)
# where columns are [case_admission_id, timestep_idx, feature_name, normalized_value]

# Find median_NIHSS feature index from the tensor itself
feature_names_in_tensor = list(X_all[0, 0, :, 2])
nihss_idx = feature_names_in_tensor.index('median_NIHSS')
print(f'median_NIHSS feature index in tensor: {nihss_idx}')
print(f'Verification - feature name: {X_all[0, 0, nihss_idx, 2]}')

# At t=0, cumulative median NIHSS = first measurement = admission NIHSS
nihss_norm = X_all[:, 0, nihss_idx, 3].astype(float)
nihss_mean = norm_params.loc[norm_params.variable == 'median_NIHSS', 'original_mean'].values[0]
nihss_std = norm_params.loc[norm_params.variable == 'median_NIHSS', 'original_std'].values[0]
admission_nihss = nihss_norm * nihss_std + nihss_mean

# Sanity check
print(f'\nAdmission NIHSS statistics:')
print(f'  Min: {admission_nihss.min():.1f}')
print(f'  Max: {admission_nihss.max():.1f}')
print(f'  Median: {np.median(admission_nihss):.1f}')
print(f'  Mean: {admission_nihss.mean():.1f}')
print(f'  IQR: {np.percentile(admission_nihss, 25):.1f} – {np.percentile(admission_nihss, 75):.1f}')

In [ ]:
# Build patient-level DataFrame
cids = X_all[:, 0, 0, 0]
end_cids = set(y_all['case_admission_id'].unique())

patient_df = pd.DataFrame({
    'case_admission_id': cids,
    'admission_nihss': np.round(admission_nihss).astype(int),
    'admission_nihss_raw': admission_nihss,
    'END': [1 if cid in end_cids else 0 for cid in cids]
})

print(f'Total patients: {len(patient_df)}')
print(f'Patients with END: {patient_df["END"].sum()}')
print(f'Patients without END: {(patient_df["END"] == 0).sum()}')

In [ ]:
def bootstrap_proportion_ci(values, n_boot=N_BOOTSTRAP, alpha=0.05, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=float)
    n = len(values)
    if n == 0:
        return (np.nan, np.nan)
    boot_props = rng.choice(values, size=(n_boot, n), replace=True).mean(axis=1)
    lower = np.percentile(boot_props, 100 * (alpha / 2))
    upper = np.percentile(boot_props, 100 * (1 - alpha / 2))
    return lower, upper

# Overall END rate
overall_n = len(patient_df)
overall_end = patient_df['END'].sum()
overall_rate = overall_end / overall_n
overall_ci = bootstrap_proportion_ci(patient_df['END'].values)
overall_median_nihss = np.median(patient_df['admission_nihss_raw'])
overall_q25 = np.percentile(patient_df['admission_nihss_raw'], 25)
overall_q75 = np.percentile(patient_df['admission_nihss_raw'], 75)

print('=== Full GSU Cohort ===')
print(f'  N patients: {overall_n}')
print(f'  Median admission NIHSS: {overall_median_nihss:.1f} (IQR {overall_q25:.1f} – {overall_q75:.1f})')
print(f'  END events: {overall_end}/{overall_n}')
print(f'  END rate: {overall_rate:.1%} (95% CI: {overall_ci[0]:.1%} – {overall_ci[1]:.1%})')

In [ ]:
# NIHSS-matched subset (matching MIMIC severity IQR: 7–19)
subset_df = patient_df[(patient_df.admission_nihss >= 7) & (patient_df.admission_nihss <= 19)]

subset_n = len(subset_df)
subset_end = subset_df['END'].sum()
subset_rate = subset_end / subset_n
subset_ci = bootstrap_proportion_ci(subset_df['END'].values)
subset_median_nihss = np.median(subset_df['admission_nihss_raw'])
subset_q25 = np.percentile(subset_df['admission_nihss_raw'], 25)
subset_q75 = np.percentile(subset_df['admission_nihss_raw'], 75)

print('=== NIHSS-Matched Subset (NIHSS 7–19) ===')
print(f'  N patients: {subset_n}')
print(f'  Median admission NIHSS: {subset_median_nihss:.1f} (IQR {subset_q25:.1f} – {subset_q75:.1f})')
print(f'  END events: {subset_end}/{subset_n}')
print(f'  END rate: {subset_rate:.1%} (95% CI: {subset_ci[0]:.1%} – {subset_ci[1]:.1%})')

In [ ]:
# NIHSS-stratified END rates
bins = [(0, 4, 'Minor (0–4)'),
        (5, 9, 'Moderate (5–9)'),
        (10, 14, 'Moderate-severe (10–14)'),
        (15, 20, 'Severe (15–20)'),
        (21, 99, 'Very severe (21+)')]

rows = []
for lo, hi, label in bins:
    mask = (patient_df.admission_nihss >= lo) & (patient_df.admission_nihss <= hi)
    grp = patient_df[mask]
    n = len(grp)
    n_end = grp['END'].sum()
    rate = n_end / n if n > 0 else 0
    ci = bootstrap_proportion_ci(grp['END'].values) if n > 0 else (np.nan, np.nan)
    rows.append({
        'NIHSS range': label,
        'N patients': n,
        'N END': n_end,
        'END rate (%)': f'{rate:.1%}',
        '95% CI': f'{ci[0]:.1%} – {ci[1]:.1%}' if n > 0 else '–'
    })

stratified_df = pd.DataFrame(rows)
print('=== NIHSS-Stratified END Rates ===')
display(stratified_df)

In [ ]:
# Summary comparison
print('=' * 70)
print('COMPARISON: GSU vs MIMIC END Rates')
print('=' * 70)
print()
print(f'Full GSU cohort:')
print(f'  N = {overall_n}')
print(f'  Median NIHSS: {overall_median_nihss:.1f} (IQR {overall_q25:.1f} – {overall_q75:.1f})')
print(f'  END rate: {overall_rate:.1%} (95% CI: {overall_ci[0]:.1%} – {overall_ci[1]:.1%})')
print()
print(f'GSU NIHSS-matched subset (7–19):')
print(f'  N = {subset_n}')
print(f'  Median NIHSS: {subset_median_nihss:.1f} (IQR {subset_q25:.1f} – {subset_q75:.1f})')
print(f'  END rate: {subset_rate:.1%} (95% CI: {subset_ci[0]:.1%} – {subset_ci[1]:.1%})')
print()
print(f'MIMIC cohort (reference):')
print(f'  N = 247')
print(f'  Median NIHSS: 13.0 (IQR 7.0 – 19.0)')
print(f'  END rate: 56.3% (139/247)')

In [ ]:
import matplotlib.pyplot as plt
from scipy import stats

# --- Data ---
mimic_end, mimic_n = 139, 247
mimic_rate = mimic_end / mimic_n
mimic_ci = bootstrap_proportion_ci(np.array([1]*mimic_end + [0]*(mimic_n - mimic_end)))

gsu_rate = subset_rate
gsu_ci = subset_ci
gsu_n_end, gsu_n_total = subset_end, subset_n

# --- Two-proportion test ---
table = np.array([[gsu_n_end, gsu_n_total - gsu_n_end],
                  [mimic_end, mimic_n - mimic_end]])
_, p_value, _, _ = stats.chi2_contingency(table, correction=False)

# --- Figure ---
fig, ax = plt.subplots(figsize=(5, 4.5))

cohorts = ['GSU\n(NIHSS 7–19)', 'MIMIC']
rates = [gsu_rate * 100, mimic_rate * 100]
ci_low = [gsu_ci[0] * 100, mimic_ci[0] * 100]
ci_high = [gsu_ci[1] * 100, mimic_ci[1] * 100]
yerr_low = [r - lo for r, lo in zip(rates, ci_low)]
yerr_high = [hi - r for r, hi in zip(rates, ci_high)]
colors = ['#4878CF', '#D65F5F']

bars = ax.bar(cohorts, rates, width=0.5, color=colors, edgecolor='black',
              linewidth=0.8, zorder=3)
ax.errorbar(cohorts, rates, yerr=[yerr_low, yerr_high],
            fmt='none', ecolor='black', capsize=6, capthick=1.5, linewidth=1.5, zorder=4)

# Annotations inside bars
for bar, r, n_end, n_tot in zip(bars, rates,
                                 [gsu_n_end, mimic_end],
                                 [gsu_n_total, mimic_n]):
    ax.text(bar.get_x() + bar.get_width() / 2, r / 2,
            f'{r:.1f}%\n({n_end}/{n_tot})',
            ha='center', va='center', fontsize=11, fontweight='bold', color='white')

# Significance bracket
y_bracket = max(ci_high) + 4
ax.plot([0, 0, 1, 1], [y_bracket - 1, y_bracket, y_bracket, y_bracket - 1],
        color='black', linewidth=1.2)
p_str = f'p < 0.0001' if p_value < 0.0001 else f'p = {p_value:.4f}'
ax.text(0.5, y_bracket + 0.8, p_str, ha='center', va='bottom', fontsize=10)

ax.set_ylabel('END Rate (%)', fontsize=12)
ax.set_ylim(0, y_bracket + 8)
ax.set_title('END Rate: GSU (NIHSS-matched) vs MIMIC', fontsize=12, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=10)

# Subtitle with median NIHSS
ax.text(0.5, -0.12,
        f'GSU subset: median NIHSS {subset_median_nihss:.0f} (IQR {subset_q25:.0f}–{subset_q75:.0f}), '
        f'MIMIC: median NIHSS 13 (IQR 7–19)',
        ha='center', va='top', fontsize=8.5, color='grey',
        transform=ax.transAxes)

plt.tight_layout()
plt.savefig('/home/klug/opsum/meta_data/short_term_outcomes/end_rate_gsu_vs_mimic.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'\nFigure saved to /home/klug/opsum/meta_data/short_term_outcomes/end_rate_gsu_vs_mimic.png')